[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/raikar8/ML-Lab/blob/main/notebooks/03_triton_matmul_int8.ipynb)

# 3 · Tiled matmul and int8 weight-only matmul (why W8A16 speeds up decoding)
Weeks 15 and 23. A tiled GEMM reuses each loaded tile many times (compute-bound at large M). At decode time M is tiny (batch 1–16),
so the GEMM becomes a memory-bound matrix–vector product — halving weight bytes with int8 (dequantised in registers) should give up
to ~2× even though the arithmetic is unchanged.

In [ ]:
# Colab: Runtime → Change runtime type → GPU (a T4 is enough except where noted). Kaggle: Settings → Accelerator.
import os, sys
if not os.path.exists("ML-Lab") and not os.path.exists("gpu"):
    !git clone --depth 1 https://github.com/raikar8/ML-Lab.git
if os.path.exists("ML-Lab"):
    %cd ML-Lab
sys.path.insert(0, ".")
import torch, triton
print(torch.__version__, triton.__version__, torch.cuda.get_device_name() if torch.cuda.is_available() else "NO GPU")
assert torch.cuda.is_available(), "Switch the runtime to a GPU"

In [ ]:
import triton.testing
def bench(fn):  # median milliseconds, with warm-up and L2 flushing handled by Triton
    return triton.testing.do_bench(fn, warmup=25, rep=100)

In [ ]:
from gpu import triton_kernels as tk
a, b = torch.randn(2048, 2048, device="cuda", dtype=torch.float16), torch.randn(2048, 2048, device="cuda", dtype=torch.float16)
torch.testing.assert_close(tk.matmul(a, b), a @ b, atol=0.5, rtol=1e-2)
for BM, BN, BK in [(32, 32, 32), (64, 64, 32), (128, 64, 32), (128, 128, 32), (64, 128, 64)]:
    t = bench(lambda: tk.matmul(a, b, BM, BN, BK))
    print(f"tile {BM}x{BN}x{BK}: {t:.3f} ms = {2 * 2048**3 / t / 1e9:.1f} TFLOP/s")
print(f"cuBLAS: {2 * 2048**3 / bench(lambda: a @ b) / 1e9:.1f} TFLOP/s")

In [ ]:
K, N = 4096, 4096
W = torch.randn(K, N, device="cuda", dtype=torch.float16) * 0.02
Wq, s = tk.quantize_w8(W)
print(f"{'M':>4} {'fp16 ms':>8} {'w8 ms':>7} {'speed-up':>9} {'rel. error':>11}")
for M in [1, 4, 16, 64, 256]:
    x = torch.randn(M, K, device="cuda", dtype=torch.float16)
    t16, t8 = bench(lambda: x @ W), bench(lambda: tk.w8_matmul(x, Wq, s, BM=16 if M <= 16 else 32))
    err = ((tk.w8_matmul(x, Wq, s) - x @ W).float().norm() / (x @ W).float().norm()).item()
    print(f"{M:>4} {t16:>8.3f} {t8:>7.3f} {t16 / t8:>8.2f}x {err:>11.4f}")

## Your turn
1. Add `@triton.autotune` with a few configs to `_matmul_kernel` and compare with cuBLAS.
2. Implement **int4** weights (two values per byte, unpack with shifts/masks in the kernel, group-wise scales of 64) — the
   W4A16 kernel behind GPTQ/AWQ deployment. Measure speed-up at M = 1.
3. Explain the M at which int8 stops helping, using the roofline from week 21.